# 01 · Ingestion & Chunking Strategy

**Graded point (Task 3): describe and justify the chunking strategy.**

We ingest a full multi-source, legal-first knowledge base (public-domain texts,
CC-BY-SA Wikipedia, our own dress-code definitions, distilled trend cards) into one
metadata-tagged collection. Reference-only copyrighted books are **never stored** — a
human/LLM distills their rules in our own words into card files.

## Chunking strategy

Two strategies, chosen per source in `data/kb/manifest.yaml`:

- **atomic** — *one style rule / concept per chunk* (authored dress codes, trend cards,
  distilled harmony rules). The chunk **is** the retrieval + citation unit.
- **section** — section-level splitting for long articles / 19th-c public-domain prose,
  so retrieval doesn't return whole-article mush.

### Justification
1. **Atomic rules make metadata-filtering precise** — each carries
   `{source, url, layer, rule_id}` plus structured fields (`occasion, formality, temp_band,
   season`), so the L4 layer is a clean structured lookup and every generator choice cites a
   stable `rule_id`.
2. **Section chunks keep long sources usable** without dominating similarity.
3. We deliberately accept the resulting **chunk-size heterogeneity** (short rules vs longer
   prose) and *measure* its effect — that is the Task-6 'change one variable' experiment in
   notebook 02.

Metadata is stamped **at ingest** — citations are impossible to bolt on later.

In [1]:
from whattowear.ingest.build_kb import ingest_all
from collections import Counter

chunks = ingest_all()   # load + chunk every ingestable source (offline-safe; Wikipedia needs network)
print('total chunks:', len(chunks))
print('per layer :', dict(Counter(c.metadata['layer'] for c in chunks)))
print('per granularity:', dict(Counter(c.metadata.get('granularity') for c in chunks)))

INFO    whattowear.ingest.build_kb: source: Wikipedia: Color theory                              layer=L1  loader=wiki_md   status=have
INFO    whattowear.ingest.build_kb:   -> 48 chunk(s) [section]
INFO    whattowear.ingest.build_kb: source: Wikipedia: Color harmony                             layer=L1  loader=wiki_md   status=have
INFO    whattowear.ingest.build_kb:   -> 12 chunk(s) [section]
INFO    whattowear.ingest.build_kb: source: Wikipedia: Complementary colors                      layer=L1  loader=wiki_md   status=have
INFO    whattowear.ingest.build_kb:   -> 38 chunk(s) [section]
INFO    whattowear.ingest.build_kb: source: Chevreul: Principles of Harmony & Contrast of Colours layer=L1  loader=epub      status=have
INFO    whattowear.ingest.build_kb:   -> 45 chunk(s) [section]
INFO    whattowear.ingest.build_kb: source: Munsell: A Color Notation                            layer=L1  loader=epub      status=have
INFO    whattowear.ingest.build_kb:   -> 39 chunk(s) [section]
INFO

total chunks: 391
per layer : {'L1': 197, 'L2': 21, 'L3': 108, 'L4': 65}
per granularity: {'section': 346, 'atomic': 45}


### A sample **atomic** chunk (one rule, fully tagged)

In [2]:
atomic = next(c for c in chunks if c.metadata.get('granularity') == 'atomic' and c.metadata['layer']=='L4')
print(atomic.metadata)
print()
print(atomic.page_content)

{'source': 'Our dress-code definitions', 'url': 'internal://l4_dresscodes', 'layer': 'L4', 'rule_id': 'L4-dc-casual', 'license': 'own', 'occasion': 'default', 'formality': 'casual', 'granularity': 'atomic'}

Casual dress code: everyday comfort-first clothing. Jeans, chinos, T-shirts, casual knits, sneakers or clean flats are all acceptable. No formality expectations; prioritize the wearer's comfort and personal expression.


### A sample **section** chunk (long PD prose, stable synthesized rule_id)

In [3]:
section = next((c for c in chunks if c.metadata.get('granularity') == 'section'), None)
if section:
    print(section.metadata)
    print(section.page_content[:400], '...')
else:
    print('No section chunks — Wikipedia/EPUB not loaded (run with network).')

{'source': 'Wikipedia: Color theory', 'url': 'https://en.wikipedia.org/wiki/Color_theory', 'layer': 'L1', 'license': 'CC-BY-SA', 'rule_id': 'L1-wikipedia-color-theory-sec-000', 'granularity': 'section'}
# Color theory

![Johann Wolfgang von Goethe's color wheel from his 1810 Theory of Colours](Color theory - Wikipedia_files/GoetheFarbkreis.jpg)

*Johann Wolfgang von Goethe's color wheel from his 1810 Theory of Colours*

**Color theory**, or more specifically **traditional color theory**, is a historical body of knowledge describing the behavior of colors — namely in color mixing, color contrast e ...


### Metadata completeness (grounding depends on this)

In [4]:
required = ('source','url','layer','rule_id')
missing = [c.metadata for c in chunks if not all(c.metadata.get(k) for k in required)]
print('chunks missing required metadata:', len(missing))
assert not missing, 'every chunk must be citable'
print('OK — every chunk is citable; rule_ids are unique by construction.')

chunks missing required metadata: 0
OK — every chunk is citable; rule_ids are unique by construction.


**Conclusion.** The KB is one collection, layer-tagged, every chunk citable. Atomic rules
are the citation unit; section chunks keep long sources usable. Heterogeneity is intentional
and measured next.